In [1]:
# ----------------------------------------------------------- 

# CELL 1: Connect to the Chinook Database 

# sqlite3 → Python's built-in library for SQLite databases 

# We connect to the .sqlite file stored in the data/ folder 

# ----------------------------------------------------------- 

 

import sqlite3 

 

# Connect to the Chinook database 

conn = sqlite3.connect('../data/Chinook_Sqlite.sqlite') 

cursor = conn.cursor() 

 

print('Connected to Chinook database successfully!') 

Connected to Chinook database successfully!


In [2]:
# ----------------------------------------------------------- 

# CELL 2: Extract All Table Names 

# sqlite_master is a special SQLite table that stores schema info 

# We filter by type='table' to get only tables, not views 

# ----------------------------------------------------------- 

 

# Query to get all table names 

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'") 

tables = [row[0] for row in cursor.fetchall()] 

 

print('Tables in Chinook database:') 

for table in tables: 

    print(f'  - {table}')

Tables in Chinook database:
  - Album
  - Artist
  - Customer
  - Employee
  - Genre
  - Invoice
  - InvoiceLine
  - MediaType
  - Playlist
  - PlaylistTrack
  - Track


In [3]:
# ----------------------------------------------------------- 

# CELL 3: Extract Column Info for Each Table 

# PRAGMA table_info() → SQLite command that returns column details 

# Returns: column id, name, data type, nullable, default, primary key 

# We store everything in a dictionary for easy access later 

# ----------------------------------------------------------- 

 

schema = {}  # dictionary to store table -> columns mapping 

 

for table in tables: 

    cursor.execute(f'PRAGMA table_info({table})')  

    columns = cursor.fetchall() 

    schema[table] = [(col[1], col[2]) for col in columns] 

    # col[1] = column name, col[2] = data type 

 

print('Schema extracted successfully!') 

print(f'Total tables: {len(schema)}') 

Schema extracted successfully!
Total tables: 11


In [4]:
# ----------------------------------------------------------- 

# CELL 4: Display the Full Schema 

# Loop through each table and print its columns and data types 

# ----------------------------------------------------------- 

 

for table, columns in schema.items(): 

    print(f'Table: {table}') 

    for col_name, col_type in columns: 

        print(f'    {col_name} ({col_type})') 

    print() 

Table: Album
    AlbumId (INTEGER)
    Title (NVARCHAR(160))
    ArtistId (INTEGER)

Table: Artist
    ArtistId (INTEGER)
    Name (NVARCHAR(120))

Table: Customer
    CustomerId (INTEGER)
    FirstName (NVARCHAR(40))
    LastName (NVARCHAR(20))
    Company (NVARCHAR(80))
    Address (NVARCHAR(70))
    City (NVARCHAR(40))
    State (NVARCHAR(40))
    Country (NVARCHAR(40))
    PostalCode (NVARCHAR(10))
    Phone (NVARCHAR(24))
    Fax (NVARCHAR(24))
    Email (NVARCHAR(60))
    SupportRepId (INTEGER)

Table: Employee
    EmployeeId (INTEGER)
    LastName (NVARCHAR(20))
    FirstName (NVARCHAR(20))
    Title (NVARCHAR(30))
    ReportsTo (INTEGER)
    BirthDate (DATETIME)
    HireDate (DATETIME)
    Address (NVARCHAR(70))
    City (NVARCHAR(40))
    State (NVARCHAR(40))
    Country (NVARCHAR(40))
    PostalCode (NVARCHAR(10))
    Phone (NVARCHAR(24))
    Fax (NVARCHAR(24))
    Email (NVARCHAR(60))

Table: Genre
    GenreId (INTEGER)
    Name (NVARCHAR(120))

Table: Invoice
    InvoiceId 

In [5]:
# ----------------------------------------------------------- 

# CELL 5: Format Schema as a Prompt-Ready String 

# This string will be inserted into the LLM prompt so the model 

# knows what tables and columns exist in the database 

# ----------------------------------------------------------- 

 

def format_schema_for_prompt(schema): 

    schema_str = '' 

    for table, columns in schema.items(): 

        schema_str += f'Table: {table}\n' 

        for col_name, col_type in columns: 

            schema_str += f'  - {col_name} ({col_type})\n' 

        schema_str += '\n' 

    return schema_str 

 

schema_string = format_schema_for_prompt(schema) 

print('Schema formatted for LLM prompt:') 

print(schema_string) 

Schema formatted for LLM prompt:
Table: Album
  - AlbumId (INTEGER)
  - Title (NVARCHAR(160))
  - ArtistId (INTEGER)

Table: Artist
  - ArtistId (INTEGER)
  - Name (NVARCHAR(120))

Table: Customer
  - CustomerId (INTEGER)
  - FirstName (NVARCHAR(40))
  - LastName (NVARCHAR(20))
  - Company (NVARCHAR(80))
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))
  - SupportRepId (INTEGER)

Table: Employee
  - EmployeeId (INTEGER)
  - LastName (NVARCHAR(20))
  - FirstName (NVARCHAR(20))
  - Title (NVARCHAR(30))
  - ReportsTo (INTEGER)
  - BirthDate (DATETIME)
  - HireDate (DATETIME)
  - Address (NVARCHAR(70))
  - City (NVARCHAR(40))
  - State (NVARCHAR(40))
  - Country (NVARCHAR(40))
  - PostalCode (NVARCHAR(10))
  - Phone (NVARCHAR(24))
  - Fax (NVARCHAR(24))
  - Email (NVARCHAR(60))

Table: Genre
  - GenreId (INTEGER)
  - Name (NVARCHAR(120